# Human-in-the-Loop: Pre-Action Gates, Risk Tiering, and Audit Logging

Ang README para sa araling ito ay nagpapakilala ng Human-in-the-Loop gamit ang isang maikling snippet na nagtatanong sa gumagamit ng `APPROVE` o `REJECT` matapos makagawa ng tugon ang ahente. Ang pattern na iyon ay isang mahusay na panimulang punto, ngunit kadalasang kailangan ng production HITL implementations ng tatlong karagdagang bagay:

1. Isang **pre-action gate** na tumatakbo **bago** isagawa ng ahente ang isang mapanganib na hakbang, upang ang gastos, hindi na maibabalik na aksyon, at pagkaantala ay manatiling kontrolado.
2. **Risk tiering**, upang ang mga mababang-panganib na aksyon ay awtomatikong maisagawa, ang mga katamtamang-panganib na aksyon ay aprubadong batch, at ang tanging mataas na panganib na mga aksyon lamang ang humahadlang sa isang tao.
3. Isang **audit log pati na rin ang revision loop**, upang bawat gate decision ay naitatala bilang JSONL, at ang pagtanggi ay muling humihiling sa ahente na magbigay ng isang istrukturadong dahilan sa halip na basta lang mag-print ng `Revising...`.

Ang notebook na ito ay bumubuo ng bawat isa sa mga ito sa ibabaw ng parehong mga primitives gaya ng `06-system-message-framework.ipynb`. Ito ay tumatakbo ng end-to-end sa `DEMO_MODE = True` (hindi kailangan ng interactive na input) o gamit ang tunay na `input()` prompts kapag `DEMO_MODE = False`. Tandaan: sa DEMO_MODE, ang pag-retry para sa ikatlong layunin ay na-script upang makita ang mekanika ng loop ng end-to-end. Ang tunay na revision-driven re-classification ay nangangailangan ng `DEMO_MODE = False` at isang operator.

**Hindi sakop (hinawakan sa ibang mga aralin):** authentication at access control (Lesson 06 README threat 2), tool-call middleware (Lesson 14 MAF deep dive), multi-agent debate patterns.

In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

token = os.environ.get("GITHUB_TOKEN", "")
if not token:
    raise RuntimeError(
        "GITHUB_TOKEN environment variable is not set. This notebook needs "
        "a GitHub PAT with 'Models: Read-only' permission. Set GITHUB_TOKEN "
        "in your environment or a local .env file before running."
    )

endpoint = "https://models.github.ai/inference"
model_name = "gpt-4o"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


## Pattern 1: Pre-action gate

Tinatawag muna ng HITL snippet sa README ang agent, tapos hiningian ng user ng pag-apruba ang output. Ito ay isang **post-action** na daloy. Naitakbo na ang agent, kaya nabayaran na ang tawag sa LLM, at nangyari na ang anumang side effect (napadalang email, naisulat sa database, nailathalang komento).

Isang **pre-action** na daloy ang naglalagay ng gate bago tumakbo ang agent sa delikadong hakbang. Ipinapanukala ng agent ang aksyon, nagpapasya ang gate kung itutuloy ito, at sa pag-apruba lang nangyayari ang side effect.

| Aspeto | Post-action approval (README snippet) | Pre-action gate (this notebook) |
|---|---|---|
| Kailan nagpapatakbo ng apruba? | Pagkatapos makagawa ng output ang agent | Bago tumakbo ang anumang side-effect |
| Gastos sa LLM kapag tinanggihan | Nababayaran na | Binabayaran lang para sa panukala, hindi sa aksyon |
| Hindi mababalik na side effect | Posible (nangyari na ang aksyon) | Naipipigil |
| Kalinawan sa audit | Ang apruba ay isang print statement | Ang apruba ay isang JSON record na may timestamp, aksyon, dahilan |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Pattern 2: Pag-uuri ng panganib

Hindi lahat ng aksyon ay nangangailangan ng pag-apruba ng tao. Ang pagbabasa lamang mula sa isang pampublikong API ay may ibang antas ng panganib kumpara sa pagpapadala ng email sa customer. Ang parehong pagtrato ay nag-aaksaya ng pansin ng operator at nagpapabagal sa ahente.

Isang simpleng modelo na may 3 antas:

| Antas | Mga Halimbawa | Daloy ng pag-apruba |
|---|---|---|
| `mababa` (read-only) | Maghanap sa knowledge base, tingnan ang mga opsyon sa flight, kunin ang isang pampublikong web page | Awtomatikong isinasagawa, naitatala para sa audit |
| `katamtaman` (murang pagbabago) | I-cache ang resulta, dagdagan ang counter, mag-iskedyul ng paalala | Awtomatikong isinasagawa, ngunit nire-review araw-araw nang pinagsama-sama |
| `mataas` (nakaharap sa panlabas o hindi na mababawi) | Magpadala ng email, singilin ang card, mag-post sa isang pampublikong channel | Hinarang hanggang sa pag-apruba ng tao |

Ito ay isang uri ng tiering. Ang mga production system ay madalas gumagamit ng mas detalyadong antas (hal., mga antas ng pahintulot sa AWS IAM, mga tier ng pag-access batay sa tungkulin). Ang bersyon na may 3 antas sa ibaba ay ang pinakamaliit na kapaki-pakinabang na bersyon para sa isang ahente na pinaghalong read-only at mga aksyon na may epekto.

Ginagamit ng classifier sa ibaba ang mga keyword heuristics upang manatiling deterministic at mura ang demo. Sa isang production system, papalitan mo ito ng isang natutunang classifier o policy engine.

In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## Pattern 3: Audit log at revision loop

Ang `print("Response approved.")` ay hindi audit log. Para sa tiwala, ang bawat gate decision ay dapat maitala bilang isang structured event na maaari mong i-query, i-replay, o i-attach sa isang incident review.

Dalawang bahagi:

1. **Append-only JSONL.** Isang linya kada desisyon, kasama ang timestamp, action, tier, decision, dahilan. Madaling i-grep, madaling ipadala sa isang totoong log store mamaya.
2. **Revision loop sa pagtanggi.** Kapag ang gate ay nagbalik ng `deny`, ang agent ay muling pinapasagot ang sarili gamit ang dahilan ng pagtanggi sa konteksto, upang ang susunod na panukala ay maiwasan ang problema.

In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.complete(
        model=model_name,
        messages=[SystemMessage(content=system), UserMessage(content=user_text)],
    )
    return response.choices[0].message.content.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}


In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## Karagdagang mga mapagkukunan

Ilang iba pang pampublikong proyekto ang nagpapatupad ng mga baryasyon ng mga pattern na HITL na ito. Ihambing ang mga pamamaraan upang makita kung ano ang akma sa iyong stack:

- **LangChain** mga human-in-the-loop tool wrappers ([docs](https://python.langchain.com/docs/integrations/tools/human_tools)): drop-in tool wrappers na humihinto ng pagpapatupad para sa input ng tao.
- **AutoGen** `UserProxyAgent` ([v0.2 docs](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ inayos uli ito): gumagamit ng papel ng ahente na partikular upang kumatawan sa tao sa multi-agent na pag-uusap.
- **Semantic Kernel** function filters ([docs](https://learn.microsoft.com/en-us/semantic-kernel/concepts/enterprise-readiness/filters)): middleware na tumatakbo sa paligid ng bawat pagtawag ng function, angkop para sa pagpapatupad ng gating logic.

Bawat proyekto ay iba ang paghawak sa tatlong sub-pattern: LangChain ay ini-wrap bilang mga tool, AutoGen ay gumagamit ng papel ng ahente, Semantic Kernel ay gumagamit ng mga middleware filter. Basahin ang isa o dalawang implementasyon mula simula hanggang katapusan bago pumili ng disenyo para sa iyong sariling ahente.

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Pagtatanggi**:
Ang dokumentong ito ay isinalin gamit ang serbisyo ng AI translation na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagama't nagsusumikap kami para sa katumpakan, pakatandaan na ang awtomatikong pagsasalin ay maaaring maglaman ng mga pagkakamali o hindi pagkakatugma. Ang orihinal na dokumento sa orihinal nitong wika ang dapat ituring na pangunahing sanggunian. Para sa mahahalagang impormasyon, inirerekomenda ang propesyonal na pagsasalin ng tao. Hindi kami mananagot sa anumang maling pagkakaintindi o maling interpretasyon na nagmula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
